Preprocessing — Member 2: Encoding and Scaling

This notebook loads Member 3's selected 18-feature train and test datasets.

We encode categorical features and standardize numerical features. The encoder and scaler are fitted only on training data, then applied to test data.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

from src.preprocessing.encoding_scaling import (
    CATEGORICAL_COLUMNS,
    build_preprocessor,
)

X_train = pd.read_csv(
    project_root / "data/processed/X_train_selected.csv"
)
X_test = pd.read_csv(
    project_root / "data/processed/X_test_selected.csv"
)

assert list(X_train.columns) == list(X_test.columns)
assert "Revenue" not in X_train.columns
assert X_train.shape[1] == 18

print("Train before encoding:", X_train.shape)
print("Test before encoding:", X_test.shape)

Train before encoding: (9764, 18)
Test before encoding: (2441, 18)


1. Encode and Scale the Selected Features

The selected dataset excludes ProductRelated, ProductRelated_Duration, and Region.

We identify categorical columns from the available features and one-hot encode them. Numerical columns are standardized. Unknown categories are handled by the encoder.

These processed CSVs are exports. Model cross-validation uses unencoded features and fits fresh preprocessing inside each fold.

In [2]:
selected_categorical_columns = [
    column for column in CATEGORICAL_COLUMNS
    if column in X_train.columns
]

selected_numeric_columns = [
    column for column in X_train.columns
    if column not in selected_categorical_columns
]

preprocessor = build_preprocessor(
    selected_numeric_columns,
    selected_categorical_columns,
)

train_array = preprocessor.fit_transform(X_train)
test_array = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    train_array,
    columns=feature_names,
    index=X_train.index,
)

X_test_processed = pd.DataFrame(
    test_array,
    columns=feature_names,
    index=X_test.index,
)

assert X_train_processed.columns.equals(X_test_processed.columns)
assert np.isfinite(train_array).all()
assert np.isfinite(test_array).all()

print("Categorical features:", len(selected_categorical_columns))
print("Numerical features:", len(selected_numeric_columns))
print("Train after encoding:", X_train_processed.shape)
print("Test after encoding:", X_test_processed.shape)

Categorical features: 7
Numerical features: 11
Train after encoding: (9764, 72)
Test after encoding: (2441, 72)


2. Save the Updated Outputs

We save the processed train and test datasets and the fitted preprocessor, replacing the previous exports.

This preprocessor expects the selected 18 features with the earlier log transformations already applied. It is separate from the GaussianNB candidate, which uses its own 17-feature preprocessing pipeline.

Transforming test features here does not train a classifier or evaluate test performance.

In [3]:
from joblib import dump

processed_folder = project_root / "data/processed"
models_folder = project_root / "models"

processed_folder.mkdir(parents=True, exist_ok=True)
models_folder.mkdir(parents=True, exist_ok=True)

X_train_processed.to_csv(
    processed_folder / "X_train_processed.csv",
    index=False,
)

X_test_processed.to_csv(
    processed_folder / "X_test_processed.csv",
    index=False,
)

dump(
    preprocessor,
    models_folder / "encoding_scaling.joblib",
)

print("Saved updated processed train and test CSVs.")
print("Saved updated encoding_scaling.joblib.")

Saved updated processed train and test CSVs.
Saved updated encoding_scaling.joblib.
